# SatQuery AI — BigEarthNet.txt Dataset Exploration
**Problem Statement:** SIH26167 (Remote Sensing Visual Question Answering)

This notebook inspects the **BigEarthNet.txt** benchmark dataset for Earth observation vision-language tasks, addressing:
1. Schema and structure of a single training example.
2. Corresponding Sentinel-1 (SAR) and Sentinel-2 (Multispectral) image files.
3. Representative Remote-Sensing VQA question/answer types.
4. Task representation (VQA, Captioning, Referring Expressions).
5. Train / Validation / Test split distribution.

In [ ]:
import json
from pathlib import Path
import pandas as pd

# Load representative sample annotations
sample_file = Path("../data/samples/bigearthnet_sample_annotations.json")
with open(sample_file, "r", encoding="utf-8") as f:
    annotations = json.load(f)

print(f"Loaded {len(annotations)} representative BigEarthNet.txt sample records.")

## 1. What does one training example look like?
A single BigEarthNet.txt record links geographical coordinates, acquisition date, Corine Land Cover (CLC) labels, natural language captions, VQA pairs, and referring expressions to co-registered Sentinel-1 and Sentinel-2 patches.

In [ ]:
example = annotations[0]
print(json.dumps(example, indent=2))

## 2. What image files correspond to an annotation?
- **Sentinel-2 (Optical/MSI):** 12 spectral bands (B01-B12, 10m/20m/60m resolutions) covering VNIR/SWIR.
- **Sentinel-1 (SAR):** 2 dual-polarization bands (VV, VH) captured via C-band Synthetic Aperture Radar.

### File Mapping:
- Optical Patch: `<s2_patch>/<band_name>.tif`
- SAR Patch: `<s1_patch>/<s1_patch>_VV.tif`, `<s1_patch>/<s1_patch>_VH.tif`

In [ ]:
for item in annotations:
    print(f"Patch ID:   {item['patch_id']}")
    print(f"S2 Optical: {item['s2_patch']}")
    print(f"S1 SAR:     {item['s1_patch']}")
    print(f"Country:    {item['country']} | Projection: {item['projection']}")
    print("-" * 60)

## 3. What VQA examples exist in the dataset?
BigEarthNet.txt provides several question-answering categories:
- **Scene Description:** Open-ended global context queries.
- **Binary Presence:** `Yes/No` detection of specific LULC classes.
- **Attribute Identification:** Specific vegetation or urban characteristics.
- **Spatial Relations:** Proximity between landmarks (e.g. forest adjacent to water bodies).

In [ ]:
vqa_rows = []
for item in annotations:
    for q in item["vqa"]:
        vqa_rows.append({
            "Patch ID": item["patch_id"],
            "Question": q["question"],
            "Answer": q["answer"],
            "Task Type": q["task_type"]
        })

df_vqa = pd.DataFrame(vqa_rows)
df_vqa

## 4. How are tasks represented?
The dataset unifies 3 major Earth Observation vision-language tasks:
1. **Visual Question Answering (VQA)** (`item["vqa"]`)
2. **Geographically Anchored Captioning** (`item["captions"]`)
3. **Referring Expressions & Localization** (`item["referring_expressions"]`)

In [ ]:
for item in annotations:
    print(f"=== Tasks for {item['patch_id']} ===")
    print(f"Caption: {item['captions'][0]}")
    print(f"VQA Count: {len(item['vqa'])}")
    print(f"Referring Expression: {item['referring_expressions'][0]['expression']}")
    print()

## 5. What are the train/validation/test splits?
BigEarthNet.txt follows standard non-overlapping geographical splits across European countries (approx 70% Train, 15% Validation, 15% Test) to ensure zero spatial data leakage.

In [ ]:
splits_summary = [{"Patch ID": item["patch_id"], "Split": item["split"], "Country": item["country"]} for item in annotations]
pd.DataFrame(splits_summary)